In [2]:
import pandas as pd

In [3]:
# Load dataset from Feature Selected csv
df = pd.read_csv("./datasets/processedTrainDataset.csv")
#df = pd.read_csv("./datasets/FeatureSelectedDataset.csv")
#df = pd.read_csv("datasets/TestDatasetExample.csv",  index_col=False)
#df = pd.read_csv("datasets/TrainDataset2025.csv",  index_col=False)
#df = pd.read_csv("datasets/FinalTestDataset2025", index_col = False)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Columns: 120 entries, pCR (outcome) to original_ngtdm_Strength
dtypes: float64(120)
memory usage: 371.4 KB


In [4]:
# 1. Check how many NaNs are in the column
rfs_nan_count = df['RelapseFreeSurvival (outcome)'].isna().sum()
print(f"Number of missing RFS values: {rfs_nan_count}")

# 2. View the rows with NaN values (optional, to see which samples are affected)
nan_rows = df[df['RelapseFreeSurvival (outcome)'].isna()]
print(nan_rows.head())

Number of missing RFS values: 76
     pCR (outcome)  RelapseFreeSurvival (outcome)  Age  ER  PgR  HER2  \
320            NaN                            NaN  NaN NaN  NaN   NaN   
321            NaN                            NaN  NaN NaN  NaN   NaN   
322            NaN                            NaN  NaN NaN  NaN   NaN   
323            NaN                            NaN  NaN NaN  NaN   NaN   
324            NaN                            NaN  NaN NaN  NaN   NaN   

     TrippleNegative  ChemoGrade  Proliferation  HistologyType  ...  \
320              NaN         NaN            NaN            NaN  ...   
321              NaN         NaN            NaN            NaN  ...   
322              NaN         NaN            NaN            NaN  ...   
323              NaN         NaN            NaN            NaN  ...   
324              NaN         NaN            NaN            NaN  ...   

     original_glszm_SmallAreaHighGrayLevelEmphasis  \
320                                            

In [5]:
from  sklearn.model_selection import train_test_split

X = df.drop(['pCR (outcome)', 'RelapseFreeSurvival (outcome)'], axis=1)
y_class = df['pCR (outcome)']
y_reg = df['RelapseFreeSurvival (outcome)']


# Remove rows where 'RelapseFreeSurvival (outcome)' is NaN     resetting the index is good practice after dropping rows

dff = df.dropna(subset=['RelapseFreeSurvival (outcome)']).reset_index(drop=True)

# Re-define X and y using the clean dataframe
X = dff.drop(['pCR (outcome)', 'RelapseFreeSurvival (outcome)'], axis=1)
y_class = dff['pCR (outcome)']
y_reg = dff['RelapseFreeSurvival (outcome)']

# Now perform the split
X_train, X_test, y_train, y_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_class, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_class
)


# Machine Learning Method Development

### Pre-Method Processing

1. Verify dataset
2. Sample Dataset
3. Separate Features from Outcomes
4. Train-Test(-Validation) Split

#### 1. Verify Dataset

In [6]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Columns: 120 entries, pCR (outcome) to original_ngtdm_Strength
dtypes: float64(120)
memory usage: 371.4 KB


,pCR (outcome),RelapseFreeSurvival (outcome),Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
count,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,...,3.200000e+02,3.200000e+02,3.200000e+02,3.200000e+02,3.200000e+02,3.200000e+02,3.200000e+02,3.200000e+02,3.200000e+02,3.200000e+02
mean,0.218750,55.736458,51.893750,0.553125,0.393750,0.293750,0.328125,2.403125,1.581250,1.134375,...,1.250000e-11,6.249991e-12,1.875000e-11,-2.187501e-11,-7.187499e-11,5.000001e-11,7.812501e-11,-3.125009e-12,-6.249992e-12,-4.687501e-11
std,0.414046,28.975211,10.851576,0.497948,0.489346,0.456192,0.470266,0.503894,0.763361,0.341589,...,1.001566e+00,1.001566e+00,1.001566e+00,1.001566e+00,1.001566e+00,1.001566e+00,1.001566e+00,1.001566e+00,1.001566e+00,1.001566e+00
min,0.000000,0.000000,26.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,...,-2.879077e+00,-2.896012e+00,-2.547060e+00,-1.438683e+00,-9.548645e-01,-1.023658e+00,-8.250463e-01,-1.201039e+00,-8.015878e-01,-9.492931e-01
25%,0.000000,38.000000,45.000000,0.000000,0.000000,0.000000,0.000000,2.000000,1.000000,1.000000,...,-6.028045e-01,-6.165424e-01,-5.688065e-01,-8.319705e-01,-7.157242e-01,-8.121921e-01,-6.023532e-01,-8.751946e-01,-7.571474e-01,-6.316149e-01
50%,0.000000,52.000000,51.000000,1.000000,0.000000,0.000000,0.000000,2.000000,1.000000,1.000000,...,-6.813296e-02,-7.621390e-02,5.219224e-02,-1.139459e-01,-1.989498e-01,-2.220624e-01,-2.384334e-01,-1.594394e-01,-2.856897e-01,-1.853705e-01
75%,0.000000,74.000000,60.000000,1.000000,1.000000,1.000000,1.000000,3.000000,2.000000,1.000000,...,6.525293e-01,6.768781e-01,7.497370e-01,6.431045e-01,1.899800e-01,5.165209e-01,3.633824e-02,6.358611e-01,2.768818e-01,1.060400e-01
max,1.000000,144.000000,79.000000,1.000000,1.000000,1.000000,1.000000,3.000000,3.000000,2.000000,...,2.708079e+00,2.766068e+00,2.118699e+00,2.700126e+00,3.711031e+00,3.647845e+00,4.304126e+00,3.026218e+00,3.467128e+00,3.923889e+00


#### 2. Sample Dataset
Generative Models:
- Variational Autoencoder (VAE) - Used in Dimensionality Reduction
- Generative Adversarial Network (GAN):#####

These do ____

##### VAE

##### GAN
A GAN is used to generate more X_train and y_train data points (synthetic data) if the original dataset was too small. This is done by training a Generator to output realistic (X, y) pairs that fool a Discriminator.

Here, given that the Example dataset only had three samples, it was deemed beneficial to use.

Implementation strategy to predict pCR and RelapseFreeSurvival (RFS) is:

- Train C-GAN: Train the model to understand the relationship between your features and your labels (pCR,RFS).

- Data Augmentation: Use the trained Generator to create synthetic patients with specific outcomes (e.g., "Generate 500 patients who had pCR=1" or "Generate patients with RFS=120 months").

- Prediction: Combine the real data with this synthetic data to train a robust supervised model (like XGBoost or MLP) that performs the actual prediction.

Implementation Steps

This code implements a Tabular C-GAN using TensorFlow/Keras. It conditions the generation on both the classification target (pCR) and the regression target (RFS).

In [7]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, Input
from sklearn.preprocessing import MinMaxScaler

In [8]:
targets = ['pCR (outcome)', 'RelapseFreeSurvival (outcome)']
feature_cols = [c for c in df.columns if c not in targets and c != 'ID']

X = df[feature_cols].values
y_pcr = df['pCR (outcome)'].values.reshape(-1, 1)
y_rfs = df['RelapseFreeSurvival (outcome)'].values.reshape(-1, 1)

# 2. Scaling
# Features to [-1, 1] (Matches Generator tanh output)
scaler_x = MinMaxScaler((-1, 1))
X_scaled = scaler_x.fit_transform(X)

# RFS (Continuous) to [0, 1] (Matches Neural Net preference)
scaler_rfs = MinMaxScaler((0, 1))
y_rfs_scaled = scaler_rfs.fit_transform(y_rfs)

# Concatenate Targets into one Condition Vector [pCR, RFS]
# Shape: (N_samples, 2)
y_conditions = np.hstack((y_pcr, y_rfs_scaled))

In [12]:
# Building Conditional GAN

latent_dim = 20     # Size of random noise vector
n_features = X.shape[1]
n_conditions = 2    # pCR + RFS

# --- Generator ---
# Inputs: Noise + Conditions (Labels)
# Output: Synthetic Features
def build_generator():
    noise_in = Input(shape=(latent_dim,))
    label_in = Input(shape=(n_conditions,))
    
    # Merge inputs
    merged = layers.Concatenate()([noise_in, label_in])
    
    x = layers.Dense(128)(merged)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.Dense(256)(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.BatchNormalization()(x)
    
    # Output matches feature dimensions
    # Tanh used to bound output between [-1, 1]
    out = layers.Dense(n_features, activation='tanh')(x)
    
    model = models.Model([noise_in, label_in], out, name="Generator")
    return model

# --- Discriminator ---
# Inputs: Features + Conditions (Labels)
# Output: Real (1) vs Fake (0)
def build_discriminator():
    feat_in = Input(shape=(n_features,))
    label_in = Input(shape=(n_conditions,))
    
    merged = layers.Concatenate()([feat_in, label_in])
    
    x = layers.Dense(256)(merged)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.4)(x)
    
    x = layers.Dense(128)(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.4)(x)
    
    out = layers.Dense(1, activation='sigmoid')(x)
    
    model = models.Model([feat_in, label_in], out, name="Discriminator")
    model.compile(loss='binary_crossentropy', optimizer=optimizers.Adam(0.0002, 0.5), metrics=['accuracy'])
    return model

# --- Build & Compile ---
generator = build_generator()
discriminator = build_discriminator()

# Create Combined GAN
discriminator.trainable = False
noise_in = Input(shape=(latent_dim,))
label_in = Input(shape=(n_conditions,))
fake_features = generator([noise_in, label_in])
validity = discriminator([fake_features, label_in])

cgan = models.Model([noise_in, label_in], validity)
cgan.compile(loss='binary_crossentropy', optimizer=optimizers.Adam(0.0002, 0.5))

discriminator.trainable = True

In [13]:
# Training loop
epochs = 500
batch_size = 32
half_batch = batch_size // 2

for epoch in range(epochs):
    # ---------------------
    #  Train Discriminator
    # ---------------------
    # Select random real samples
    idx = np.random.randint(0, X_scaled.shape[0], half_batch)
    real_features = X_scaled[idx]
    real_labels = y_conditions[idx]
    
    # Generate fake samples
    noise = np.random.normal(0, 1, (half_batch, latent_dim))
    fake_features = generator.predict([noise, real_labels], verbose=0)
    
    # Labels: 1 for real, 0 for fake
    valid_y = np.ones((half_batch, 1))
    fake_y = np.zeros((half_batch, 1))
    
    # Train D
    d_loss_real = discriminator.train_on_batch([real_features, real_labels], valid_y)
    d_loss_fake = discriminator.train_on_batch([fake_features, real_labels], fake_y)
    d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)
    
    # ---------------------
    #  Train Generator
    # ---------------------
    # We want the discriminator to think these fakes are real (label=1)
    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    
    # Sample random conditions to train generator on
    idx_gen = np.random.randint(0, X_scaled.shape[0], batch_size)
    sampled_labels = y_conditions[idx_gen]
    
    valid_y = np.ones((batch_size, 1))
    
    # Train G
    g_loss = cgan.train_on_batch([noise, sampled_labels], valid_y)
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch}: D Loss: {d_loss[0]:.4f} | G Loss: {g_loss:.4f}")

Epoch 0: D Loss: nan | G Loss: nan
Epoch 100: D Loss: nan | G Loss: nan
Epoch 200: D Loss: nan | G Loss: nan
Epoch 300: D Loss: nan | G Loss: nan
Epoch 400: D Loss: nan | G Loss: nan


In [ ]:
# Using C-GAN for downstream prediction

# 1. Define what kind of data you want to generate
# e.g., Balanced pCR (50% class 0, 50% class 1) with random RFS
num_synthetic = 1000
noise = np.random.normal(0, 1, (num_synthetic, latent_dim))

# Create synthetic conditions
syn_pcr = np.random.choice([0, 1], num_synthetic).reshape(-1, 1)
# Generate RFS randomly from the original distribution range (0 to 1)
syn_rfs_scaled = np.random.rand(num_synthetic, 1) 

syn_conditions = np.hstack((syn_pcr, syn_rfs_scaled))

# 2. Generate Features
syn_features_scaled = generator.predict([noise, syn_conditions])
syn_features = scaler_x.inverse_transform(syn_features_scaled)
syn_rfs = scaler_rfs.inverse_transform(syn_rfs_scaled)

# 3. Create Augmented Training Set
X_aug = np.vstack((X, syn_features))
y_pcr_aug = np.vstack((y_pcr, syn_pcr))
y_rfs_aug = np.vstack((y_rfs, syn_rfs))

# 4. Train your Final Predictor (e.g., MLP)
from sklearn.neural_network import MLPClassifier, MLPRegressor

# Classifier for pCR
clf = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500)
clf.fit(X_aug, y_pcr_aug.ravel())

# Regressor for RFS
reg = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500)
reg.fit(X_aug, y_rfs_aug.ravel())

print("Predictors trained on Augmented Data.")

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


ValueError: Input X contains NaN.
MLPClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

VAE-GAN
#- Possibly try VAE-GAN to remove issue with gaussian distribution

#### 3. Separate Features from Outcomes
(data is from csv)

In [ ]:
# Separating features and outcomes


X = df.drop(['pCR (outcome)', 'RelapseFreeSurvival (outcome)'], axis=1)
y_class = df['pCR (outcome)']
y_reg = df['RelapseFreeSurvival (outcome)']

NameError: name 'df' is not defined

#### 4. Train-test-Validation split
where do we do cross validation, with what methods to generate error

In [ ]:
# Train test split
from sklearn.model_selection import train_test_split

# Split Data
X_train, X_test, y_train, y_test, y_clf_train, y_clf_test = train_test_split(X, y_reg, y_class, test_size=0.2, random_state = 42, stratify= y_class)


### Machine Learning Models

##### Linear:
- Multivariate Linear Regression
- ~~Logistic Regression~~ (Took over 10 min until interrupted) | Classification
- Naïve Bayes (don't forget different types) | Classification
- K-Nearest Neighbour (KNN) | Classification
- Support Vector Classifier/Regressor

##### Non-Linear:
- Kernel Support Vector Classifier/Regressior
- Decision Tree Classifier/Regressor
- Random Forest Classifier/Regressor

##### Deep Learning Models:
- Multilayer Perceptron Classifier/Regressor
- Convolution Neural Networks -------------------------------------------------

### Classification

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier


results = {}

##### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression
log_reg = LogisticRegression(max_iter=100)
log_reg.fit(X_train, y_clf_train)
results['Logistic Regression Acc'] = log_reg.score(X_test, y_clf_test)

C:\Users\EK\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


##### Naïve Bayes
Different types: gaussian, bernoulli, (counts), (another one I forgot)

In [ ]:
# Naïve Bayes (Gaussian)
nb = GaussianNB()
nb.fit(X_train, y_clf_train)
results['Naive Bayes Acc'] = nb.score(X_test, y_clf_test)

##### K Nearest Neighbour
Value of K hyperparameter

In [ ]:
# KNN
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_clf_train)
results['KNN Acc'] = knn.score(X_test, y_clf_test)

##### Support Vector Classifier
Hard Margin, soft margin

In [ ]:
# Hard Margin SVM (Approximated with large C)
svm_hard = SVC(kernel='linear', C=100)
svm_hard.fit(X_train, y_clf_train)
results['Hard Margin SVM Acc'] = svm_hard.score(X_test, y_clf_test)

##### Kernel SVC

In [ ]:
# Kernel SVM (RBF)
svm_kernel = SVC(kernel='rbf')
svm_kernel.fit(X_train, y_clf_train)
results['Kernel SVM Acc'] = svm_kernel.score(X_test, y_clf_test)

##### Decision Tree
Reasons for params

In [ ]:
# Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train, y_clf_train)
results['Decision Tree Acc'] = dt.score(X_test, y_clf_test)

##### Random Forest
Reasons for params

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=10)
rf.fit(X_train, y_clf_train)
results['Random Forest Acc'] = rf.score(X_test, y_clf_test)

##### Multi-Layer Perceptron Classifier
Explain Parameters

In [ ]:
# MLP
mlp = MLPClassifier(hidden_layer_sizes=(50, 50), max_iter=500)
mlp.fit(X_train, y_clf_train)
results['MLP Acc'] = mlp.score(X_test, y_clf_test)

In [ ]:
print("Model execution successful. Results on dummy test set:")
print(results)

Model execution successful. Results on dummy test set:
{'Naive Bayes Acc': 0.1375, 'KNN Acc': 0.75, 'Kernel SVM Acc': 0.775, 'Decision Tree Acc': 0.6875, 'Random Forest Acc': 0.7625, 'MLP Acc': 0.775, 'Logistic Regression Acc': 0.7625}


### Regression

##### Multivariate Linear Regression
explain params

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
print(f"Linear Regression Score: {lin_reg.score(X_test, y_test)}")

Linear Regression Score: -2254.2453237337036


##### Support Vector Regressor
Explain parameters

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

# Kernel SVR (RBF)
svr = SVR(kernel='rbf')
svr.fit(X_train, y_train)
y_pred_svr = svr.predict(X_test)

print(f"SVR MSE: {mean_squared_error(y_test, y_pred_svr)}")

SVR MSE: 677.3637249391907


##### Kernel SVR

##### Decision Tree Regressor
Explain Parameters

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt_reg = DecisionTreeRegressor(random_state=42)
dt_reg.fit(X_train, y_train)
y_pred_dt = dt_reg.predict(X_test)

print(f"Decision Tree MSE: {mean_squared_error(y_test, y_pred_dt)}")

Decision Tree MSE: 1985.0733506779725


##### Random Forest Regressor
Explain Parameters

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)
y_pred_rf = rf_reg.predict(X_test)

print(f"Random Forest MSE: {mean_squared_error(y_test, y_pred_rf)}")

Random Forest MSE: 710.6563212945834


##### Multi Layer Perceptron Regressor
Explain Parameters

In [ ]:
from sklearn.neural_network import MLPRegressor

# MLP with 2 hidden layers
mlp_reg = MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
mlp_reg.fit(X_train, y_train)
y_pred_mlp = mlp_reg.predict(X_test)

print(f"MLP MSE: {mean_squared_error(y_test, y_pred_mlp)}")

MLP MSE: 103234667229.52676


# K Fold Cross Validation
Create pipeline and use k fold 

# Method Evaluation

### Classification
- Confusion Matrix
- Precision/Recall/Specificity/Sensitivity
- (Balanced) Classification Accuracy

### Regression
- Mean Square Error
- etc

In [ ]:
# Export chosen Model (using pickle?)
# Go to "final notebooks"